In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==0.21.1

# Structure Refinement: LMO, ECHIDNA

This example refines an LMO structure with Li/Ni site mixing against
constant-wavelength neutron powder diffraction data collected on the
ECHIDNA diffractometer at ANSTO. The workflow starts from approximate
structural and profile parameters, constrains the coupled site
occupancies, and performs a Rietveld refinement.

## 🛠️ Import Library

In [2]:
import easydiffraction as edi

## 📦 Define Project

The project manages the structure, experiment, analysis, and saved
results used throughout the tutorial.

### Create Project

In [3]:
project = edi.Project(
    name='lmo_echidna',
    description='LMO refinement using ECHIDNA neutron powder diffraction data.',
)

### Save Initial Project

Create the project directory before fitting so that analysis results
can be written as they are produced.

In [4]:
project.save_as(dir_path='projects/refine-lmo-echidna')

Saving project 📦 'lmo_echidna' to '../../../projects/refine-lmo-echidna'


├── 📄 project.edi
├── 📁 structures/
├── 📁 experiments/
├── 📁 analysis/
│   └── 📄 analysis.edi
└── 📁 reports/
    └── 📄 lmo_echidna.html


## 🧩 Define Structure

The rhombohedral LMO model contains two crystallographic cation sites.
Li1 and Ni1 share the site at z = 1/2, while Li2 and Ni2 share the site
at z = 0. Their starting occupancies describe a small amount of Li/Ni
site mixing.

### Create Structure from CIF

Define the complete starting structure in a compact inline CIF. The
hexagonal setting of space group R-3m is used, with approximate cell
dimensions and oxygen z coordinate.

In [5]:
structure_cif = """
data_lmo

_cell.length_a  2.88
_cell.length_b  2.88
_cell.length_c 14.18
_cell.angle_alpha 90.
_cell.angle_beta  90.
_cell.angle_gamma 120.

_space_group.name_h_m "R -3 m"
_space_group.coord_system_code h

loop_
_atom_site.id
_atom_site.type_symbol
_atom_site.fract_x
_atom_site.fract_y
_atom_site.fract_z
_atom_site.occupancy
_atom_site.adp_iso
_atom_site.adp_type
O   O   0. 0. 0.26  1.0000  0.94645 Biso
Ni1 Ni  0. 0. 0.5   0.0184  1.00000 Biso
Li1 Li  0. 0. 0.5   0.9816  1.00000 Biso
Li2 Li  0. 0. 0.0   0.0184  1.00000 Biso
Ni2 Ni  0. 0. 0.0   0.9816  1.00000 Biso
"""

In [6]:
project.structures.add_from_cif_str(structure_cif)

In [7]:
project.structures.show_names()

Defined structures 🧩


['lmo']


Use a short alias to access the structure parameters below.

In [8]:
structure = project.structures['lmo']

### Display Structure

Inspect the structure as text and as an interactive crystal model.

In [9]:
structure.show_as_text()

Structure 🧩 'lmo' as text


,Edi
1,data_lmo
2,
3,_cell.length_a 2.88
4,_cell.length_b 2.88
5,_cell.length_c 14.18
6,_cell.angle_alpha 90.
7,_cell.angle_beta 90.
8,_cell.angle_gamma 120.
9,
10,"_space_group.name_h_m ""R -3 m"""


In [10]:
project.display.structure(struct_name='lmo')

Structure 🧩 'lmo' (Atom view type: 'covalent')


## 🔬 Define Experiment

Load the measured pattern, choose the calculation engine, configure
the instrument and peak profile, and link the structure to the data.

### Download Data

Download the LMO pattern from the EasyDiffraction online data
repository. The columns contain 2-theta, intensity, and the standard
uncertainty of the measured intensity.

In [11]:
data_path = edi.download_data('meas-lmo-echidna', destination='data')

Getting data...


Data 'meas-lmo-echidna': LMO, ECHIDNA (ANSTO), wavelength 1.6215 A


✅ Data 'meas-lmo-echidna' downloaded to '../../../data/meas-lmo-echidna.dat'


### Create Experiment

In [12]:
project.experiments.add_from_data_path(
    name='echidna',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
)

Data loaded successfully


Experiment 🔬 'echidna'. Number of data points: 3200.


Use a short alias to access the experiment parameters below.

In [13]:
experiment = project.experiments['echidna']

### Select Calculator

Use the CrysFML calculation engine for this refinement.

In [14]:
experiment.calculator.show_supported()

Calculator types


,,Type,Description
1,,crysfml,CrysFML library for crystallographic calculations
2,*,cryspy,CrysPy library for crystallographic calculations


In [15]:
experiment.calculator.type = 'crysfml'

Calculator for experiment 'echidna' changed to


crysfml


### Set Instrument

Set the measured neutron wavelength and approximate calibration
corrections for the 2-theta zero, sample displacement, and sample
transparency.

In [16]:
experiment.instrument.setup_wavelength = 1.6215
experiment.instrument.calib_twotheta_offset = 0.0
experiment.instrument.calib_sample_displacement = 0.03
experiment.instrument.calib_sample_transparency = 0.02

### Set Peak Profile

Select the Thompson-Cox-Hastings pseudo-Voigt profile. U, V, and W
define its Gaussian broadening; Y defines its Lorentzian broadening;
and the Finger-Cox-Jephcoat terms describe the low-angle asymmetry.

In [17]:
experiment.peak.show_supported()

Peak types


,,Type,Description
1,*,pseudo-voigt,CWL pseudo-Voigt profile
2,,thompson-cox-hastings,CWL Thompson-Cox-Hastings profile with FCJ asymmetry correction.


In [18]:
experiment.peak.type = 'thompson-cox-hastings'

⚠️ Switching peak profile type adds these settings with defaults:
• asym_fcj_1=0.0
• asym_fcj_2=0.0


Peak profile type for experiment 'echidna' changed to


thompson-cox-hastings


In [19]:
experiment.peak.broad_gauss_u = 0.1
experiment.peak.broad_gauss_v = -0.3
experiment.peak.broad_gauss_w = 0.4
experiment.peak.broad_lorentz_y = 0.1
experiment.peak.asym_fcj_1 = 0.08
experiment.peak.asym_fcj_2 = 0.08

### Set Absorption

Apply the Hewat cylindrical-sample absorption correction with an
approximate value of the dimensionless absorption-radius product.

In [20]:
experiment.absorption.type = 'cylinder-hewat'
experiment.absorption.mu_r = 0.3

Absorption type changed to


cylinder-hewat


### Set Excluded Regions

Exclude the low- and high-angle regions outside the useful measured
range from 12 to 162 degrees.

In [21]:
experiment.excluded_regions.create(id='1', start=0.0, end=12.0)
experiment.excluded_regions.create(id='2', start=162.0, end=180.0)

### Set Background

Estimate initial background points from the measured pattern.

In [22]:
experiment.background.show_supported()

Background types


,,Type,Description
1,,chebyshev,Chebyshev polynomial background
2,*,line-segment,Linear interpolation between points


In [23]:
experiment.background.auto_estimate()

In [24]:
experiment.background.show()

Line-segment background points


,Position,Intensity
1,12.02420,558.74541
2,20.03970,640.62497
3,32.05130,497.74000
4,39.42070,641.87000
5,58.71960,553.94103
6,93.37010,550.51000
7,149.84400,653.58448
8,161.99680,772.38099


### Set Linked Structure

Link the LMO model to the experiment and provide an initial estimate
for its scale factor.

In [25]:
experiment.linked_structures.create(structure_id='lmo', scale=10.0)

### Inspect Experiment

Display the configured experiment as text.

In [26]:
experiment.show_as_text()

Experiment 🔬 'echidna' as text


,Edi
1,data_echidna
2,
3,_experiment_type.sample_form powder
4,"_experiment_type.beam_mode ""constant wavelength"""
5,_experiment_type.radiation_probe neutron
6,_experiment_type.scattering_type bragg
7,
8,_diffrn.ambient_temperature ?
9,_diffrn.ambient_pressure ?
10,_diffrn.ambient_magnetic_field ?


## 🚀 Perform Analysis

Inspect the starting calculation, constrain the coupled site-mixing
parameters, select the independent refinement parameters, and fit the
model to the measured pattern.

### Display Initial Pattern

In [27]:
project.display.pattern(expt_name='echidna')

### Set Constraints

First create readable aliases for the displacement and occupancy
parameters involved in the constraints.

In [28]:
project.analysis.aliases.create(
    id='biso_Li1',
    param=structure.atom_sites['Li1'].adp_iso,
)
project.analysis.aliases.create(
    id='biso_Li2',
    param=structure.atom_sites['Li2'].adp_iso,
)
project.analysis.aliases.create(
    id='biso_Ni1',
    param=structure.atom_sites['Ni1'].adp_iso,
)
project.analysis.aliases.create(
    id='biso_Ni2',
    param=structure.atom_sites['Ni2'].adp_iso,
)

project.analysis.aliases.create(
    id='occ_Li1',
    param=structure.atom_sites['Li1'].occupancy,
)
project.analysis.aliases.create(
    id='occ_Li2',
    param=structure.atom_sites['Li2'].occupancy,
)
project.analysis.aliases.create(
    id='occ_Ni1',
    param=structure.atom_sites['Ni1'].occupancy,
)
project.analysis.aliases.create(
    id='occ_Ni2',
    param=structure.atom_sites['Ni2'].occupancy,
)

Atoms sharing a crystallographic site use the same Biso value. The
occupancy constraints keep each shared site fully occupied and couple
the same Li/Ni exchange fraction across both sites. Consequently,
`occ_Li1` is the only independent occupancy parameter.

In [29]:
project.analysis.constraints.create(
    id='1',
    expression='biso_Ni1 = biso_Li1',
)
project.analysis.constraints.create(
    id='2',
    expression='biso_Li2 = biso_Ni2',
)
project.analysis.constraints.create(
    id='3',
    expression='occ_Ni1 = 1 - occ_Li1',
)
project.analysis.constraints.create(
    id='4',
    expression='occ_Li2 = 1 - occ_Li1',
)
project.analysis.constraints.create(
    id='5',
    expression='occ_Ni2 = occ_Li1',
)

In [30]:
project.analysis.constraints.show()

User defined constraints


,id,expression
1,1,biso_Ni1 = biso_Li1
2,2,biso_Li2 = biso_Ni2
3,3,occ_Ni1 = 1 - occ_Li1
4,4,occ_Li2 = 1 - occ_Li1
5,5,occ_Ni2 = occ_Li1


Constraints enabled: True


### Set Free Parameters

Refine the two independent cell lengths, oxygen z coordinate, the two
independent cation Biso values, and the independent Li occupancy.

In [31]:
structure.cell.length_a.free = True
structure.cell.length_c.free = True

structure.atom_sites['O'].fract_z.free = True
structure.atom_sites['Li1'].adp_iso.free = True
structure.atom_sites['Ni2'].adp_iso.free = True
structure.atom_sites['Li1'].occupancy.free = True

Refine the scale, instrument calibration terms, U/V/W/Y profile terms,
and active background-point intensities. The asymmetry and absorption
parameters remain fixed at their approximate values.

In [32]:
experiment.linked_structures['lmo'].scale.free = True

experiment.instrument.calib_twotheta_offset.free = True
experiment.instrument.calib_sample_displacement.free = True
experiment.instrument.calib_sample_transparency.free = True

experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.peak.broad_gauss_w.free = True
experiment.peak.broad_lorentz_y.free = True

for point in experiment.background:
    point.intensity.free = True

Display all parameters selected for refinement.

In [33]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,lmo,cell,,length_a,2.88000,,-inf,inf,Å
2,lmo,cell,,length_c,14.18000,,-inf,inf,Å
3,lmo,atom_site,O,fract_z,0.26000,,-inf,inf,
4,lmo,atom_site,Li1,occupancy,0.98160,,-inf,inf,
5,lmo,atom_site,Li1,adp_iso,1.00000,,-inf,inf,Å²
6,lmo,atom_site,Ni2,adp_iso,1.00000,,-inf,inf,Å²
7,echidna,linked_structure,lmo,scale,10.00000,,-inf,inf,
8,echidna,peak,,broad_gauss_u,0.10000,,-inf,inf,deg²
9,echidna,peak,,broad_gauss_v,-0.30000,,-inf,inf,deg²
10,echidna,peak,,broad_gauss_w,0.40000,,-inf,inf,deg²


### Select Minimizer

Use the Levenberg-Marquardt optimizer provided by Bumps.

In [34]:
project.analysis.minimizer.show_supported()

Minimizer types


,,Type,Description
1,,bumps,BUMPS library using the default Levenberg-Marquardt method
2,,bumps (amoeba),BUMPS library with Nelder-Mead simplex method
3,,bumps (de),BUMPS library with differential evolution method
4,,bumps (dream),BUMPS library with DREAM Bayesian sampling
5,,bumps (lm),BUMPS library with Levenberg-Marquardt method
6,,dfols,DFO-LS library for derivative-free least-squares optimization
7,,emcee,emcee affine-invariant ensemble Bayesian sampling
8,,lmfit,LMFIT library using the default Levenberg-Marquardt method
9,,lmfit (least_squares),LMFIT library with SciPy's trust region reflective algorithm
10,*,lmfit (leastsq),LMFIT library with Levenberg-Marquardt least squares method


In [35]:
project.analysis.minimizer.type = 'bumps (lm)'

⚠️ Switching minimizer type removes these settings:
• gradient_tolerance


Current minimizer changed to


bumps (lm)


### Fit Model

In [36]:
project.analysis.minimizer.chi_square_change_tolerance = 1e-2

In [37]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'echidna' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.23,84.57,
2,24,4.88,57.98,31.4% ↓
3,47,9.63,6.22,89.3% ↓
4,70,14.31,2.89,53.5% ↓
5,93,19.55,2.44,15.5% ↓
6,116,25.23,2.41,1.2% ↓
7,141,39.86,2.41,


🏆 Best goodness-of-fit (reduced χ²) is 2.41 at iteration 141


✅ Fitting complete.


### Inspect Results

Review the fit statistics, refined parameters, and parameter
correlations, then compare the refined calculation with the data.

In [38]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.
2,chi_square_change_tolerance,0.01,Relative change in the objective (chi-square) used to stop fitting.
3,parameter_change_tolerance,1e-08,Relative change in fitted parameters used to stop fitting.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),39.86
4,📏 Goodness-of-fit (reduced χ²),2.41
5,"📏 R-factor (Rf, %)",3.94
6,"📏 R-factor squared (Rf², %)",4.50
7,"📏 Weighted R-factor (wR, %)",5.02


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lmo,cell,,length_a,Å,2.8800,2.8753,0.0001,0.16 % ↓
2,lmo,cell,,length_c,Å,14.1800,14.1839,0.0004,0.03 % ↑
3,lmo,atom_site,O,fract_z,,0.2600,0.2591,0.0001,0.36 % ↓
4,lmo,atom_site,Li1,occupancy,,0.9816,0.9801,0.0018,0.15 % ↓
5,lmo,atom_site,Li1,adp_iso,Å²,1.0000,1.9525,0.1097,95.25 % ↑
6,lmo,atom_site,Ni2,adp_iso,Å²,1.0000,0.2277,0.0118,77.23 % ↓
7,echidna,linked_structure,lmo,scale,,10.0000,16.0249,0.0841,60.25 % ↑
8,echidna,peak,,broad_gauss_u,deg²,0.1000,0.0966,0.0024,3.39 % ↓
9,echidna,peak,,broad_gauss_v,deg²,-0.3000,-0.2831,0.0069,5.63 % ↓
10,echidna,peak,,broad_gauss_w,deg²,0.4000,0.4025,0.0053,0.61 % ↑


In [39]:
project.display.fit.correlations()

In [40]:
project.display.pattern(expt_name='echidna')

## 💾 Save Project

Save the refined parameters and analysis results in the project
directory created near the beginning of the tutorial.

In [41]:
project.save()

Saving project 📦 'lmo_echidna' to '../../../projects/refine-lmo-echidna'


├── 📄 project.edi
├── 📁 structures/
│   └── 📄 lmo.edi
├── 📁 experiments/
│   └── 📄 echidna.edi
├── 📁 analysis/
│   └── 📄 analysis.edi
└── 📁 reports/
    └── 📄 lmo_echidna.html
